# Task 4: One-Hot Encoding

**The Goal**: Learn to handle real-world, non-numeric factory data.

**The Action**: Introduce a categorical column to the dataset space. Fortunately, our base `ai4i2020.csv` already includes a categorical `Type` column (with qualities `L`, `M`, and `H`). We will convert this text-based label into valid numeric binary columns.

**The Research**: Explain the "Dummy Variable Trap" and how we circumvent it by dropping an encoding column.

In [4]:
import pandas as pd

# Load dataset
df = pd.read_csv('ai4i2020.csv')

# Original categorical data observation
print("Original Categorical Column ('Type') distribution:")
print(df['Type'].value_counts())

# Demonstration of standard one-hot encoding vs dummy-proofed encoding
# 1. Standard one-hot encoding (creates a column for EVERY category)
standard_encoded = pd.get_dummies(df['Type'], prefix='Type_Standard')

# 2. Dropping the first column to avoid the Dummy Variable Trap
dummy_proof_encoded = pd.get_dummies(df['Type'], prefix='Type', drop_first=True)

print("\n--- Encoding Example ---")
print("\nBefore (Original):")
print(df[['Type']].head(5))

print("\nStandard One-Hot (All categories preserved - Risks Multicollinearity):")
print(standard_encoded.head(5))

print("\nDummy Variable Trap Proofed (First category 'H' dropped):")
print(dummy_proof_encoded.head(5))

# We can reintegrate this back into our main dataframe
df_final = pd.concat([df.drop('Type', axis=1), dummy_proof_encoded], axis=1)
print("\nFinal Dataset Columns for Modeling:")
print(df_final.columns.tolist())

Original Categorical Column ('Type') distribution:
Type
L    6000
M    2997
H    1003
Name: count, dtype: int64

--- Encoding Example ---

Before (Original):
  Type
0    M
1    L
2    L
3    L
4    L

Standard One-Hot (All categories preserved - Risks Multicollinearity):
   Type_Standard_H  Type_Standard_L  Type_Standard_M
0            False            False             True
1            False             True            False
2            False             True            False
3            False             True            False
4            False             True            False

Dummy Variable Trap Proofed (First category 'H' dropped):
   Type_L  Type_M
0   False    True
1    True   False
2    True   False
3    True   False
4    True   False

Final Dataset Columns for Modeling:
['UDI', 'Product ID', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Type_L', 'Type_M']


### Research: The Dummy Variable Trap

**What is the Dummy Variable Trap?**
The "Dummy Variable Trap" inherently occurs when independent features are highly correlated—a situation specifically termed *perfect multicollinearity*. 

If you use pure one-hot encoding on a category column with 3 variables (Like Product `Type`: `L`, `M`,  `H`), the algorithm outputs 3 discrete columns: `Type_L`, `Type_M`, `Type_H`. Because each part belongs to exactly one category, the equation is statically predictable: 

$$Type\_L + Type\_M + Type\_H = 1$$

If we know that `Type_L` is 0 and `Type_M` is 0, we can mathematically guarantee that `Type_H` is 1. This means the 3rd variable provides absolutely no new information to the model; rather, it introduces a redundancy.

**Why dropping one column is essential:**
For many Machine Learning algorithms (especially Linear Regression matrices), having variables that perfectly predict one another prevents the internal linear equation from functioning correctly (because the matrix inversion formula collapses or becomes computationally unstable; also known as having a singular matrix).

To fix this, we **drop the first column** (`drop_first=True`).
- If `Type_L` = 0 and `Type_M` = 0, the model conceptually interprets this baseline as the dropped category (`Type_H`). 
- This $n-1$ methodology preserves exactly the same amount of logic without stepping into perfectly collinear paradoxes!